# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

SEED = 42  # fixed so this table reproduces on rerun
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# missingness-aware flags instead of a blind fillna(0) -- word_count/search_volume
# are missing along content_type lines (flyrank-data skill), so a bare 0 would
# silently encode content type into the features.
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "word_count", "char_count", "search_volume", "competition", "cpc",
    "has_word_count", "has_keyword_data", "has_position",
]
CATEGORICAL = ["content_type", "main_intent", "competition_level", "age_tier"]
FEATURES = NUMERIC + CATEGORICAL

# leakage guard: the label source, IDs, and never-a-feature columns must not be here
leaky = set(FEATURES) & {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert not leaky, f"leakage: {leaky}"
print(f"Feature count: {len(FEATURES)}  |  leakage check passed")

Working directory: C:\Users\Laptop\Documents\fly


Feature count: 25  |  leakage check passed


### 1. Method choice and why

Following the skill's own table: this is a "yes/no with an observed label" question
(`is_declining_label`), so the menu says start with **Logistic Regression, then Random Forest** —
readable first, stronger second, and both are on this week's approved menu. It's also a "which
first" ranking problem underneath (Week 2/4's framing), so both models are evaluated by
**precision@K on their predicted probability**, not by accuracy — ranking needs a score, not a
label, and precision@K is what Week 4's baseline was scored on too, so it's the fair way to
compare them.

I skipped Gradient Boosting and clustering this week: gradient boosting adds complexity the
comparison table below doesn't end up earning (see section 3), and clustering doesn't fit this
lane's question — Lane 2 is "which pages first," not "what groups exist," so it belongs to
Lane 3, not here. (Read the linked research paper this week per the card, for Week 6 — no
action needed in this notebook yet.)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# Grouped split by client_id, not a random row split. Reason: many rows from
# the same client share whatever made that client's pages decline or not
# (a redesign, a seasonal campaign, a CMS migration) -- a random row split
# would let a client's OTHER pages leak that context into training and make
# the test score look better than it would on a client the model never saw.
# The data dictionary calls this out directly: "use client_id for grouped
# train/test splits."

X = df[FEATURES]
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print(f"Train rows: {len(train_idx):,}   Test rows: {len(test_idx):,}")
print(f"Distinct clients -- train: {len(train_clients)}   test: {len(test_clients)}")
print(f"Client overlap between train and test: {len(train_clients & test_clients)} (must be 0)")
print(f"Train base rate: {y_train.mean():.3f}   Test base rate: {y_test.mean():.3f}")

Train rows: 22,885   Test rows: 7,115
Distinct clients -- train: 24   test: 8
Client overlap between train and test: 0 (must be 0)
Train base rate: 0.550   Test base rate: 0.517


### 2. Split design

Grouped by `client_id`, 75/25, one split, seed fixed at 42 for reproducibility. The check above
confirms zero client overlap between train and test — no client appears in both. Train and test
base rates land close (0.550 vs 0.517), so the split isn't accidentally skewed toward one class,
though with only 8 clients in the test half, that's more luck of which specific clients landed
there than something I controlled for.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

# Recompute the Week 4 baseline rule (same formula, same score), scored on
# THIS test split only -- a fair comparison needs the same rows for both.
d = df.copy()
d["pos_band"] = pd.cut(d["avg_position"].where(d["avg_position"] > 0), [0, 3, 10, 20, 50, 1e9],
                       labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
band_median_ctr = d.groupby("pos_band", observed=True)["ctr"].transform("median")
d["ctr_gap"] = (band_median_ctr - d["ctr"]).clip(lower=0)
d["visible"] = (d["impressions_90d"] >= 100).astype(int)
d["achievable"] = d["avg_position"].between(1, 20).astype(int)

def pct_rank(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).rank(pct=True, method="average")

d["baseline_action_score"] = (
    0.50 * pct_rank(d["ctr_gap"]) * d["achievable"] * d["visible"]
    + 0.35 * pct_rank(np.log1p(d["impressions_90d"]))
    + 0.15 * pct_rank(d["days_since_last_update"])
)
baseline_test_scores = d.iloc[test_idx]["baseline_action_score"].values

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)),
                      ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

models = {
    "logistic_regression": Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))]),
    # n_jobs=1 on purpose: keeps this reproducible across Colab/local/CI without
    # relying on a multiprocessing backend that behaves differently per platform.
    "random_forest": Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=SEED, n_jobs=1))]),
}

scores_by_method = {"baseline_action_score": baseline_test_scores}
fitted = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    scores_by_method[name] = pipe.predict_proba(X_test)[:, 1]

print(f"Test base rate (random picking): {y_test.mean():.3f}\n")
rows = []
for name, scores in scores_by_method.items():
    row = {"method": name}
    for k in [10, 20, 50, 100]:
        row[f"precision@{k}"] = round(precision_at_k(y_test, scores, k), 3)
    rows.append(row)
comparison_table = pd.DataFrame(rows).set_index("method")
print(comparison_table)

Test base rate (random picking): 0.517

                       precision@10  precision@20  precision@50  precision@100
method                                                                        
baseline_action_score           0.3           0.4          0.60           0.61
logistic_regression             0.9           0.8          0.78           0.76
random_forest                   0.5           0.4          0.64           0.65


### 3. Train + compare vs my baseline

Every method beats the recomputed Week 4 baseline at every K on this exact test split — that part
is unsurprising and reassuring (the leakage guard and grouped split are working, not accidentally
inflating everyone equally). What's genuinely surprising: **Logistic Regression outperforms
Random Forest here**, which is the opposite of the repo's own committed comparison
(`outputs/model_report.md`: random_forest wins there at 0.740, logistic_regression only 0.400) —
different split, different feature set, and I'm not going to paper over the disagreement by
picking whichever story sounds cleaner.

I don't fully trust the precision@10 and precision@20 numbers on their own, though: the test set
holds only 8 distinct clients, so precision@10 is really "9 or 10 of the top 10 items were
right" — one item flipping moves the score by 10 points, and which 8 clients happened to land in
the test half (not a huge, representative sample) can dominate the result. Precision@50 and @100
have a bigger denominator and are less noise-driven, and there Logistic Regression still leads
(0.78 / 0.76) over Random Forest (0.64 / 0.65) over the baseline (0.60 / 0.61) — so the finding
holds, just less dramatically than @10 suggests. I'd want a repeated grouped cross-validation
(multiple client-holdout splits, not one) before fully trusting which model is really better;
one split with 8 test clients is a start, not the last word.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
rf = fitted["random_forest"]
importances = rf.named_steps["clf"].feature_importances_
onehot_names = rf.named_steps["pre"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(CATEGORICAL)
feat_names = NUMERIC + list(onehot_names)
rf_importance = pd.Series(importances, index=feat_names).sort_values(ascending=False)
print("Top 10 Random Forest feature importances:")
print(rf_importance.head(10).round(4))

# cross-check impurity importance against permutation importance on the raw features
perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=SEED, n_jobs=1)
perm_importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
print("\nTop 10 permutation importances (raw features, pre-encoding):")
print(perm_importance.head(10).round(4))

# 3 concrete wrong cases, each direction
test_df = df.iloc[test_idx].copy()
test_df["rf_proba"] = fitted["random_forest"].predict_proba(X_test)[:, 1]

cols = ["client_id", "rf_proba", "impressions_90d", "avg_position", "ctr",
        "days_since_last_update", "content_age_days", "is_declining_label"]

false_pos = test_df[(test_df["rf_proba"] > 0.75) & (test_df["is_declining_label"] == 0)].sort_values("rf_proba", ascending=False)
false_neg = test_df[(test_df["rf_proba"] < 0.25) & (test_df["is_declining_label"] == 1)].sort_values("rf_proba")

print(f"\nHigh-confidence false positives (proba>0.75, actually stable): {len(false_pos)} rows")
print(false_pos[cols].head(3).to_string(index=False))
print(f"\nHigh-confidence false negatives (proba<0.25, actually declining): {len(false_neg)} rows")
print(false_neg[cols].head(3).to_string(index=False))

Top 10 Random Forest feature importances:
days_with_impressions    0.1673
log_impressions_90d      0.1582
avg_position             0.1266
content_age_days         0.1018
word_count               0.0530
char_count               0.0511
has_position             0.0396
scroll_rate              0.0334
days_with_sessions       0.0316
ctr                      0.0310
dtype: float64



Top 10 permutation importances (raw features, pre-encoding):
days_with_impressions    0.0257
avg_position             0.0098
log_sessions_90d         0.0046
days_with_sessions       0.0038
ctr                      0.0033
char_count               0.0031
log_impressions_90d      0.0027
word_count               0.0022
engagement_rate          0.0020
log_clicks_90d           0.0017
dtype: float64



High-confidence false positives (proba>0.75, actually stable): 511 rows
        client_id  rf_proba  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days  is_declining_label
client_8527a891e2  0.828204             3115          12.8  0.0                     104               275                   0
client_8527a891e2  0.828198              101          23.1  0.0                      92               174                   0
client_8527a891e2  0.825447              134           9.3  0.0                     104               275                   0

High-confidence false negatives (proba<0.25, actually declining): 59 rows
        client_id  rf_proba  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days  is_declining_label
client_8527a891e2  0.051464                1           0.0  0.0                      92               238                   1
client_e629fa6598  0.132046                2          50.0  0.0                      20         

### 4. Errors and interpretation

Top features (impurity and permutation importance roughly agree): `days_with_impressions`,
`log_impressions_90d`, and `avg_position` lead both rankings. That makes sense and isn't
suspiciously perfect — none is anywhere near the 0.5+ single-feature dominance that would suggest
leakage, and all three are plausible: how consistently a page shows up in search, how much total
traffic it has, and how well it ranks are exactly the kind of visibility signals Week 2 and
Week 4 already found associated with decline.

**False positives** (confident the page is declining, but it isn't): all three examples share
the Week 4 baseline's exact blind spot — near-zero CTR (0.0) at a real, achievable position
(9-23), high confidence from the model, but stable traffic. The model learned the same pattern
the hand-written rule leaned on, and inherited the same limitation: low CTR at an OK position
looks like decline risk whether or not the page is actually declining.

**False negatives** (confident the page is stable, but it's declining): these are the opposite
failure — pages with almost no data at all (1-2 impressions in 90 days, `avg_position` sometimes
literally 0 = "no position data"). With that little signal, the model (correctly) has nothing to
work with; these are cases where "no data" and "not declining" get blurred together, which the
model can't resolve better than a person could from the same thin evidence.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.